# ML · Encuentro 2 · Clasificar y comprobar alertas

Clasificar y entender los errores

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [De una puntuación a una categoría](#scrollTo=tema-ml-clasificacion)
2. [Diez plantas, cuatro resultados posibles](#scrollTo=tema-ml-metricas)
3. [La decisión tiene un umbral](#scrollTo=tema-ml-umbral)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · De una puntuación a una categoría

**Lo que vas a poder hacer:** Construir una lista de revisión a partir de puntuaciones y una regla explícita.

Una alerta puede priorizar plantas para revisión, sin reemplazar la comprobación de campo.

### Entiende la idea

La clasificación responde preguntas sobre categorías: aquí, si una planta tiene plaga. El modelo puede producir una puntuación y una regla la convierte en una categoría predicha. Esa decisión se contrasta con la etiqueta real. Separar estos tres elementos permite entender por qué un mismo modelo puede generar distintas listas de alertas al cambiar el umbral.

### Un caso pequeño antes del programa

**Decidir cuáles plantas revisar**

Seis de las diez plantas inventadas del explorador. Las puntuaciones están fijadas; aquí no entrenamos un clasificador.

| Planta | Puntuación | Realidad conocida | Alerta con umbral 0,50 |
| --- | --- | --- | --- |
| P1 | 0,90 | Con plaga | Sí |
| P3 | 0,70 | Con plaga | Sí |
| P4 | 0,40 | Con plaga | No |
| P5 | 0,75 | Sana | Sí |
| P6 | 0,55 | Sana | Sí |
| P8 | 0,20 | Sana | No |

1. Positivo significa «con plaga». La clase responde esa pregunta, no califica a la planta como buena o mala.
2. Aplicamos una regla: puntuación mayor o igual a 0,50 produce alerta. Por eso P5 sí y P4 no.
3. Contrastamos con la realidad: P5 es una falsa alerta y P4 es una plaga omitida. La regla puede equivocarse.

**Qué demuestra:** Una puntuación de 0,75 no significa 75 % de daño ni asegura presencia de plaga.

### Ahora identifica los datos de esta práctica

La curva usa un índice hídrico simulado en mm y la categoría alto rendimiento (0/1). No diagnostica plagas: las diez plantas del caso de alertas son otro conjunto.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Distingue la puntuación calculada de la categoría obtenida al aplicar un umbral.


In [ ]:
# Regresión logística — versión mínima y clara (1 sola variable, 1 sola figura)
# -----------------------------------------------------------------------------
# Objetivo didáctico:
# - Que vean la "S" (sigmoide) y entiendan que la salida es PROBABILIDAD de clase.
# - Con 1 variable (índice hídrico) para no distraer: si hay buen balance hídrico,
#   aumenta la probabilidad de “alto rendimiento”.
#
# Copia esta ÚNICA CELDA en Colab y ejecútala.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# 1) Dataset MUY simple (1 variable) ------------------------------------------
# Índice hídrico (mm): positivo = agua suficiente; negativo = déficit.
rng = np.random.default_rng(0)
n = 200
indice_hidrico = rng.normal(loc=0.0, scale=25.0, size=n)  # media 0 para tener casos positivos y negativos

# Probabilidad “verdadera” pedagógica (sigmoide con umbral ~0)
# Ajuste suave: a mayor índice, mayor probabilidad de alto rendimiento.
p_true = 1 / (1 + np.exp(-(indice_hidrico / 8.0)))  # dividir por 8 ensancha la S para verla mejor

# Etiqueta binaria a partir de esa probabilidad (simulamos la realidad con azar)
y = (rng.uniform(0, 1, n) < p_true).astype(int)

X = indice_hidrico.reshape(-1, 1)

print("Vista rápida de los datos (primeras 5 filas):")
display(pd.DataFrame({"indice_hidrico_mm": indice_hidrico[:5].round(2), "alto_rendimiento": y[:5]}))

print(f"Prevalencia de clase positiva (alto_rendimiento=1): {y.mean():.3f}")

# 2) Entrenar Regresión Logística ---------------------------------------------
# Sin estandarizar para que el coeficiente se interprete directo en mm.
logit = LogisticRegression(solver="lbfgs")
logit.fit(X, y)

b0 = float(logit.intercept_[0])
b1 = float(logit.coef_[0, 0])
print("\nModelo aprendido (forma canónica de la sigmoide):")
print("p(y=1 | x) = 1 / (1 + exp(-(b0 + b1 * x)))")
print(f"b0 = {b0:.3f}, b1 = {b1:.3f}")
print(f"Odds ratio por +1 mm en índice hídrico: exp(b1) = {np.exp(b1):.3f}")

# 3) Probabilidades de clase y predicción (umbral 0.5) ------------------------
p_hat = logit.predict_proba(X)[:, 1]
y_pred = (p_hat >= 0.5).astype(int)

acc = accuracy_score(y, y_pred)
cm = confusion_matrix(y, y_pred)
print(f"\nAccuracy de entrenamiento con umbral 0.5: {acc:.3f}")
print("Matriz de confusión (filas = real, columnas = predicho):")
print(cm)

# 4) Visualización única: datos y curva sigmoide ------------------------------
# Ordenamos por x para dibujar la sigmoide suave
idx = np.argsort(indice_hidrico)
x_plot = indice_hidrico[idx]
p_plot = p_hat[idx]

plt.figure(figsize=(7,5))
# Puntos crudos (x, y binario) con jitter vertical para ver densidad
y_jitter = y[idx] + rng.normal(0, 0.02, size=n)
plt.scatter(x_plot, y_jitter, alpha=0.4, label="Observaciones (0/1 con jitter)")

# Curva de probabilidad aprendida
plt.plot(x_plot, p_plot, linewidth=2, label="Probabilidad estimada (sigmoide)")

# Línea de decisión (p=0.5 -> b0 + b1*x = 0 -> x = -b0/b1)
if b1 != 0:
    x_threshold = -b0 / b1
    plt.axvline(x_threshold, linestyle="--", label=f"Umbral p=0.5 (x≈{x_threshold:.1f})")

plt.xlabel("Índice hídrico (mm)")
plt.ylabel("Probabilidad de alto rendimiento")
plt.title("Regresión logística: 1 variable, 1 curva, 1 idea")
plt.ylim(-0.1, 1.1)
plt.legend()
plt.show()

# 5) Mini-demostración práctica ------------------------------------------------
# ¿Qué probabilidad predice el modelo para tres escenarios?
escenarios = np.array([[-40.0], [0.0], [+40.0]])  # déficit, neutral, superávit
p_esc = logit.predict_proba(escenarios)[:, 1]
tabla = pd.DataFrame({"indice_hidrico_mm": escenarios.ravel(), "p_alto_rendimiento": p_esc.round(3)})
print("\nProbabilidades para tres escenarios simples:")
display(tabla)

# Explicación concisa en prints (para el estudiante):
print("\nResumen pedagógico:")
print("1) predict_proba estima probabilidades; predict devuelve clases con una regla de decisión.")
print("2) Esa probabilidad sigue una curva en 'S' (sigmoide).")
print("3) Para decidir clase, usamos un UMBRAL (típicamente 0.5).")
print("4) Si mueves el umbral, cambias la sensibilidad vs. especificidad (no mostrado aquí para mantenerlo simple).")
print("5) Odds = p/(1-p). exp(b1) multiplica las odds por +1 mm, no los puntos porcentuales de probabilidad.")
print("   Aquí b1>0 asocia un índice mayor con mayores odds estimadas; no demuestra causalidad.")


### La misma puntuación, tres reglas de decisión
Para x=0 mm seguimos z → p → clase y luego contamos errores en los mismos 200 ejemplos de entrenamiento. Los puntos conservan su puntuación; la línea de corte se mueve. Los triángulos son positivos reales y los círculos negativos reales; el color indica la alerta. La precisión queda no definida cuando no se emite ninguna alerta.


In [ ]:
x_consulta = 0.0
z_consulta = b0 + b1*x_consulta
p_consulta = 1/(1+np.exp(-z_consulta))
print(f'z = {b0:.6f} + {b1:.6f} × {x_consulta} = {z_consulta:.6f}')
print(f'p = 1/(1+exp(−z)) = {p_consulta:.6f}')
assert np.isclose(p_consulta, logit.predict_proba([[x_consulta]])[0,1])
filas_cortes=[]
fig, ejes=plt.subplots(1,3,figsize=(13,4),sharey=True)
for ax,corte in zip(ejes,[.2,.5,.8]):
    alerta=p_hat>=corte
    tp=int(np.sum(alerta & (y==1)));fp=int(np.sum(alerta & (y==0)))
    fn=int(np.sum((~alerta) & (y==1)));tn=int(np.sum((~alerta) & (y==0)))
    precision=tp/(tp+fp) if tp+fp else np.nan
    recall=tp/(tp+fn) if tp+fn else np.nan
    filas_cortes.append([corte,tp,fn,fp,tn,precision,recall])
    for clase,marca,nombre in [(0,'o','Real: no alto'),(1,'^','Real: alto')]:
        sel=y==clase
        ax.scatter(p_hat[sel],np.flatnonzero(sel),marker=marca,
                   c=np.where(alerta[sel],'#0075ad','#89998c'),s=18,label=nombre)
    ax.axvline(corte,color='#a34a28',ls='--',label='Corte elegido')
    ax.axvspan(corte,1,color='#0075ad',alpha=.08)
    ax.set(xlim=(0,1),xlabel='Puntuación del modelo',title=f'Corte {corte:.2f}: {int(alerta.sum())} alertas')
ejes[0].set_ylabel('Índice de la fila, no una medición');ejes[-1].legend(fontsize=8)
plt.tight_layout();plt.show()
display(pd.DataFrame(filas_cortes,columns=['Corte','TP','FN','FP','TN','Precisión','Recall']))
print('El corte lo elegimos. No mueve las puntuaciones ni cambia las etiquetas reales.')


### Lee, comprueba y explica

La curva usa un índice hídrico simulado en mm y la categoría alto rendimiento (0/1). No diagnostica plagas: las diez plantas del caso de alertas son otro conjunto.

**Prueba pequeña:** Distingue la puntuación calculada de la categoría obtenida al aplicar un umbral.

**Vuelve al caso:** Define la categoría positiva y la regla de alerta. Conserva puntuación, decisión y realidad en columnas separadas.

**Pregunta:** ¿Puntuación, clase predicha y clase real son lo mismo?

**Respuesta razonada:** No. La puntuación es una salida numérica; el umbral la convierte en clase predicha. La clase real se obtiene mediante observación o anotación y permite evaluar la decisión. Mover el umbral no cambia la realidad.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-clasificacion)


## Tema 2 · Contar aciertos, falsas alertas y omisiones

**Lo que vas a poder hacer:** Construir un reporte que muestre qué detecta y qué omite una alerta.

«Acertamos bastante» oculta si omitimos plagas o revisamos demasiadas plantas sanas.

### Entiende la idea

Las métricas de clasificación resumen decisiones comparadas con su realidad conocida. Antes de calcular porcentajes, organizamos todos los casos en una matriz de confusión con los cuatro resultados de abajo. Cada métrica elige un denominador diferente y responde una pregunta distinta. Debemos declarar que la clase positiva es «con plaga» para interpretar los conteos.

### Un caso pequeño antes del programa

**Las diez plantas, sin perder ninguna**

Las diez plantas inventadas del explorador, agrupadas por resultado con umbral 0,50. Estas filas representan todos los casos.

| Realidad | Decisión | Plantas | Cantidad |
| --- | --- | --- | --- |
| Con plaga | Alerta | P1, P2, P3 | 3 |
| Con plaga | Sin alerta | P4 | 1 |
| Sana | Alerta | P5, P6 | 2 |
| Sana | Sin alerta | P7, P8, P9, P10 | 4 |

1. Cuatro plantas tenían plaga y detectamos tres: recall = 3/4 = 75 %. La planta P4 quedó omitida.
2. De cinco alertas, tres eran correctas: precision = 3/5 = 60 %. Dos serían falsas alertas.
3. Acertamos tres positivos y cuatro negativos: accuracy = (3 + 4)/10 = 70 %. Cada métrica responde otra pregunta.

**Qué demuestra:** 70 % de aciertos no significa detectar 70 % de plagas: aquí detectamos 75 %.

### Ahora identifica los datos de esta práctica

Las filas de la matriz son la realidad y las columnas la predicción. La clase positiva es con plaga: TP=3, FP=2, FN=1, TN=4.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Precision divide entre alertas; recall entre positivos reales. No intercambies sus denominadores.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


In [ ]:
real=np.array([1,1,1,1,0,0,0,0,0,0])
pred=np.array([1,1,1,0,1,1,0,0,0,0])
cm=confusion_matrix(real,pred,labels=[0,1])
display(pd.DataFrame(cm,index=['Real sin plaga','Real con plaga'],columns=['Pred sin plaga','Pred con plaga']))
display(pd.DataFrame({'métrica':['Accuracy','Precision','Recall','F1'],'valor':[accuracy_score(real,pred),precision_score(real,pred),recall_score(real,pred),f1_score(real,pred)]}))
fig,ax=plt.subplots();ax.imshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=20)
ax.set_xticks([0,1],['Sin plaga','Con plaga']);ax.set_yticks([0,1],['Sin plaga','Con plaga']);ax.set_xlabel('Predicción');ax.set_ylabel('Realidad');ax.set_title('Matriz de confusión · 10 plantas');plt.show()


### Lectura del gráfico 2 · ¿Qué errores cometimos en diez plantas?

Esta imagen es una **matriz de conteos**, no una curva. El eje vertical indica la **realidad** y el horizontal la **predicción**. Para leer una casilla, combina su fila y su columna. El azul más oscuro significa más plantas en esa casilla, no mayor calidad.

| Realidad → predicción | Plantas | Interpretación |
|---|---:|---|
| Sin plaga → sin plaga | 4 | Sanas correctamente descartadas (TN). |
| Sin plaga → con plaga | 2 | Falsas alertas (FP). |
| Con plaga → sin plaga | 1 | Plaga que se pasó por alto (FN). |
| Con plaga → con plaga | 3 | Plagas detectadas (TP). |

La diagonal reúne los **7 aciertos**. Fuera de ella están **3 errores**, con consecuencias distintas: revisar una planta sana o dejar pasar una planta con plaga.

La tabla de métricas usa decimales: **accuracy 0,70 = 70 %** de aciertos; **precision 0,60 = 60 %** de las alertas correctas; **recall 0,75 = 75 %** de las plantas con plaga detectadas. **F1 ≈ 0,667** es su media armónica: `F1 = 2 × precision × recall / (precision + recall)`. Ninguna métrica por sí sola dice cuánto cuesta cada error. Estas diez respuestas son un ejemplo construido; aquí no entrenamos un clasificador.


### Encuentros 3 y 4: nombres completos de las medidas y métodos

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **MAE** — MAE: Mean Absolute Error (Error absoluto medio). Promedio de las diferencias sin signo entre cada predicción y su respuesta observada: se resta, se elimina el signo y se promedian las distancias. Menor es mejor sobre los mismos casos y unidades; 0 indica coincidencia en todos esos casos. Conserva la unidad de la respuesta y no significa que cada caso tenga ese mismo error. Si se observan 4 y 10 cajas y se predicen 3 y 7, los errores absolutos son 1 y 3 cajas: MAE = (1+3)/2 = 2 cajas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **RMSE** — RMSE: Root Mean Squared Error (Raíz del error cuadrático medio). Raíz cuadrada del promedio de diferencias al cuadrado entre predicción y respuesta observada. Vuelve a las unidades de la respuesta, aunque conserva la mayor influencia de errores grandes del MSE. Menor es mejor sobre los mismos casos; no es el promedio de las distancias absolutas. Con errores de 1 y 3 cajas, MSE = 5 cajas² y RMSE = √5 ≈ 2,24 cajas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)

- **R²** — R²: Coeficiente de determinación. Símbolo de una medida de regresión: R² = 1 − suma de errores cuadrados / suma de diferencias cuadradas respecto a la media observada. Para respuestas no constantes, 1 es ajuste perfecto y 0 rinde lo mismo que predecir su media; puede ser negativo. No es porcentaje de aciertos; con respuestas constantes el cociente no está definido y algunas bibliotecas sustituyen el resultado. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

- **DBSCAN** — DBSCAN: Density-Based Spatial Clustering of Applications with Noise (Agrupación espacial basada en densidad con casos no agrupados). Método que forma grupos conectando zonas con suficientes vecinos. eps fija la distancia máxima entre vecinos y min_samples el mínimo para que un punto sea núcleo, contando al propio punto. No se elige de antemano el número de grupos; −1 identifica puntos sin grupo con esa configuración. El radio no limita la distancia entre todos los miembros de un grupo y quedar aparte no demuestra mala calidad. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)


### Lee, comprueba y explica

Las filas de la matriz son la realidad y las columnas la predicción. La clase positiva es con plaga: TP=3, FP=2, FN=1, TN=4.

**Prueba pequeña:** Precision divide entre alertas; recall entre positivos reales. No intercambies sus denominadores.

**Vuelve al caso:** Escribe el numerador y el denominador de cada métrica: detectar plagas, acertar alertas y acertar decisiones son preguntas diferentes.

**Pregunta:** Si casi todas las plantas están sanas, ¿por qué la exactitud puede ocultar un problema?

**Respuesta razonada:** Un sistema que siempre responde sana acertaría muchas veces, pero detectaría cero plantas con plaga. Por eso revisamos recall, precisión y los cuatro conteos, indicando cuál es la clase positiva.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-metricas)


## Tema 3 · Ajustar alertas a cuatro revisiones

**Lo que vas a poder hacer:** Configurar una política de alertas según la capacidad de revisión.

Un equipo con pocas personas para revisar debe decidir cuánto riesgo de omisión aceptar.

### Entiende la idea

Ahora usamos las métricas para elegir una política de alertas. La regla es dar alerta si puntuación ≥ umbral: un corte más alto puede reducir revisiones y también dejar positivos sin atender. Las puntuaciones y las plantas permanecen fijas; cambian las decisiones. La elección debe responder a consecuencias y capacidad, hacerse con validación y comprobarse después en una prueba separada.

### Un caso pequeño antes del programa

**Cuatro decisiones cambian de regla**

Cuatro plantas inventadas del explorador. Cambiamos el umbral conservando puntuaciones y realidades; no generamos otro modelo.

| Planta y realidad | Puntuación | Alerta a 0,50 | Alerta a 0,70 |
| --- | --- | --- | --- |
| P3: con plaga | 0,70 | Sí | Sí |
| P4: con plaga | 0,40 | No | No |
| P5: sana | 0,75 | Sí | Sí |
| P6: sana | 0,55 | Sí | No |

1. La regla usa mayor o igual: P3 conserva su alerta cuando puntuación y umbral valen 0,70.
2. P6 pierde la alerta al subir a 0,70: evitamos esa falsa alerta. P5 sigue por encima del nuevo límite.
3. P4 continúa omitida. Subir el límite no resuelve toda equivocación; bajar a 0,40 sí activaría su alerta.

**Qué demuestra:** El umbral decide qué revisar. Elegirlo requiere conocer consecuencias y usar validación, dejando intacta la prueba final.

### Ahora identifica los datos de esta práctica

Se conservan las mismas diez puntuaciones y realidades del aula. Compara 0,30, 0,50 y 0,70 y después cambia UMBRAL.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** La regla incluye la igualdad: puntuación ≥ umbral. Cambiar el corte no entrena otro modelo; cuatro revisiones no eliminan el riesgo de omisión.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


In [ ]:
# Las mismas diez plantas del aula: 1 = con plaga, 0 = sin plaga.
# Las puntuaciones están dadas para explicar decisiones; aquí no se ajusta un modelo.
real = np.array([1,1,1,1,0,0,0,0,0,0])
puntuaciones = np.array([.90,.80,.70,.40,.75,.55,.30,.20,.10,.05])
CAPACIDAD = 4
UMBRAL = .70  # Cambia solo este valor y vuelve a ejecutar la celda.
def evaluar_corte(corte):
    alertas = puntuaciones >= corte
    return {'Umbral': corte, 'Alertas': int(alertas.sum()),
            'Falsas alertas': int(((real == 0) & alertas).sum()),
            'Omisiones': int(((real == 1) & ~alertas).sum()),
            'Cabe en 4 revisiones': bool(alertas.sum() <= CAPACIDAD)}
comparacion_umbrales = pd.DataFrame([evaluar_corte(c) for c in [.30,.50,.70]])
display(comparacion_umbrales)
assert comparacion_umbrales['Alertas'].tolist() == [7,5,4]
assert comparacion_umbrales['Falsas alertas'].tolist() == [3,2,1]
assert comparacion_umbrales['Omisiones'].tolist() == [0,1,1]
decisiones = pd.DataFrame({'Planta': ['P'+str(i+1) for i in range(10)],
    'Realidad': np.where(real == 1, 'Con plaga', 'Sin plaga'),
    'Puntuación': puntuaciones, 'Alerta': puntuaciones >= UMBRAL})
display(decisiones)
display(pd.DataFrame([evaluar_corte(UMBRAL)]))
plt.figure(figsize=(9,4))
plt.bar(decisiones.Planta, puntuaciones, color=np.where(real == 1, '#a34d32', '#567447'))
plt.axhline(UMBRAL, color='black', linestyle='--', label=f'Umbral {UMBRAL:.2f}: se incluye la igualdad')
plt.ylabel('Puntuación dada'); plt.xlabel('Planta'); plt.ylim(0,1)
plt.title('Conservar puntuaciones y cambiar decisiones'); plt.legend(); plt.show()
print('Color = realidad conocida; altura = puntuación; la línea determina la alerta.')
print('Cumplir la capacidad no demuestra que se hayan detectado todas las plagas.')


### Lee, comprueba y explica

Se conservan las mismas diez puntuaciones y realidades del aula. Compara 0,30, 0,50 y 0,70 y después cambia UMBRAL.

**Prueba pequeña:** La regla incluye la igualdad: puntuación ≥ umbral. Cambiar el corte no entrena otro modelo; cuatro revisiones no eliminan el riesgo de omisión.

**Vuelve al caso:** Elige la regla considerando revisiones necesarias, falsas alertas y omisiones; indica qué harás con los casos que no entren en la revisión.

**Pregunta:** ¿Cambiar el umbral vuelve a entrenar el modelo?

**Respuesta razonada:** En este ejemplo, no: conserva sus puntuaciones y cambia la decisión. Elegimos el corte con validación y después medimos la política elegida en una prueba separada.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-umbral)


## Cierra el encuentro con una explicación

1. **De una puntuación a una categoría:** Define la categoría positiva y la regla de alerta. Conserva puntuación, decisión y realidad en columnas separadas.
2. **Diez plantas, cuatro resultados posibles:** Escribe el numerador y el denominador de cada métrica: detectar plagas, acertar alertas y acertar decisiones son preguntas diferentes.
3. **La decisión tiene un umbral:** Elige la regla considerando revisiones necesarias, falsas alertas y omisiones; indica qué harás con los casos que no entren en la revisión.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
